# DocLens + Confidence and Degradation Check (CDC) with Qwen3-VL-8B on Kaggle

Your novelty, on the same backbone and questions as the Table 9 reproduction (`doclens_qwen_kaggle.ipynb`).

**What is compared, on identical questions and identical Lens outputs**

| Row in the report | Meaning |
|---|---|
| DocLens (always verify) | the base paper's method |
| Always abstain | Lens Module + one answer (= the paper's "w/o Reasoning Module", Table 2) |
| Always correct | the full Reasoning Module with degradation-aware correction for every question |
| **CDC** | **your gate**: abstain when confident, correct otherwise (threshold picked on a dev split) |
| Oracle gate | upper bound for any gate |

on **clean** pages and on **blurred** and **noisy** copies (MinerU re-parses the degraded pages, so the OCR text is degraded too).
The report also gives over-refinement, the abstention rate, tokens per question, the AUROC of the confidence, and a
**paired bootstrap test** (CDC minus DocLens, 95% CI) for every variant.

**Two sessions, one notebook.** Change only `STAGE` in the first code cell.

1. `STAGE = "prepare"` (about 3-4 GPU hours, no Qwen server):
   *Add Input* → the "Agentic" notebook, **Version #2** (contains `parsed_MMLongBenchDoc.tar`). **Run All**.
   It builds the degraded copies, parses them with MinerU, calibrates the detector, and packs everything into
   `novelty_variants.tar`. Then *Save Version → Quick Save* with *"Save output for this version"* turned on.
2. `STAGE = "run"`: *Add Input* → (a) the Agentic Version #2, (b) **this notebook's saved prepare version**,
   (c) **your Qwen reproduction notebook's saved version** (contains `results_qwen/.../qwen_doclens.jsonl`, reused for the
   clean pages so they are not recomputed). **Run All**. Resumes after an interruption: re-attach the inputs and Run All again.

*Settings*: **GPU T4 x2**, **Internet On**, in both sessions.

In [ ]:
# ======================= SETTINGS =======================
STAGE = "prepare"                 # "prepare" (session 1) -> then "run" (session 2)

WORK = "/kaggle/working"
CDC_REPO = "https://github.com/Mariam1543/aqqal123.git"
CDC_BRANCH = "claude/wizardly-rubin-kdthhy"
DATASET = "MMLongBenchDoc"

N_TABLE9, SEED = 200, 0           # must match doclens_qwen_kaggle.ipynb (the novelty questions are a subset)
N_NOVELTY = 100                   # questions for the CDC experiments
MAX_TOTAL_PAGES = 1500            # MinerU pages per degraded variant
VARIANTS = ["clean", "blur_s2", "noise_s2"]
CALIB_VARIANTS = ["blur_s1", "blur_s2", "blur_s3", "noise_s1", "noise_s2", "noise_s3", "jpeg_s3", "lowres_s2"]

MAX_MODEL_LEN, NAV_CHUNK, MAX_IMAGES, MAX_CONCURRENT = 40960, 20, 48, 4   # same as the Table 9 notebook
PAPER_PROMPTS = True
PILOT_QUESTIONS = 3
assert STAGE in ("prepare", "run")

## 1. Get the code

In [ ]:
import os, sys, subprocess, shutil, json, glob, random, time, re, tarfile
from pathlib import Path
os.chdir(WORK)
if not os.path.exists("DocLens"):
    subprocess.run(["git", "clone", "https://github.com/dwzhu-pku/DocLens.git"], check=True)
shutil.rmtree("doclens_cdc", ignore_errors=True)
shutil.rmtree("_cdc_repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", CDC_BRANCH, CDC_REPO, "_cdc_repo"], check=True)
shutil.copytree("_cdc_repo/doclens_cdc", "doclens_cdc")
shutil.rmtree("_cdc_repo")
DOCLENS, CDC = f"{WORK}/DocLens", f"{WORK}/doclens_cdc"
ROOT = f"{WORK}/full_{DATASET}"           # clean/ comes from parsed_MMLongBenchDoc.tar
CALIB_ROOT = f"{WORK}/calib_{DATASET}"
RESULTS = f"{WORK}/results_novelty/{DATASET}"
os.makedirs(RESULTS, exist_ok=True)
sys.path.insert(0, CDC)
DEGRADED = [v for v in VARIANTS if v != "clean"]
DEG_STR, CALIB_STR = " ".join(DEGRADED), " ".join(CALIB_VARIANTS)
print("stage:", STAGE)

## 2. Install (prepare: MinerU · run: vLLM)

In [ ]:
if STAGE == "prepare":
    !pip install -q "mineru[core]==2.7.6" 2>&1 | tail -n 3
    !mineru --version
else:
    !pip install -q "vllm>=0.11.0" 2>&1 | tail -n 3
!pip install -q -r {CDC}/requirements-kaggle.txt 2>&1 | tail -n 2

## 3. Data, the Table 9 questions and the novelty subset

In [ ]:
def find_input(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    return hits[0] if hits else None

tar_clean = find_input("parsed_MMLongBenchDoc.tar")
assert tar_clean, "Add the Agentic notebook's Version #2 (parsed_MMLongBenchDoc.tar) as an input."
if not os.path.exists(f"{ROOT}/clean/data/{DATASET}/documents"):
    !tar -xf {tar_clean} -C {WORK}
!python {CDC}/scripts/download_data.py --dataset {DATASET} --out {DOCLENS}/data
SAMPLES = f"{DOCLENS}/data/{DATASET}/samples.json"
DOCUMENTS = f"{DOCLENS}/data/{DATASET}/documents"

# exactly the same 200 questions as doclens_qwen_kaggle.ipynb
all_q = json.load(open(SAMPLES))
picked = set(random.Random(SEED).sample(range(len(all_q)), N_TABLE9))
TABLE9 = f"{ROOT}/samples_random{N_TABLE9}_seed{SEED}.json"
json.dump([q for i, q in enumerate(all_q) if i in picked], open(TABLE9, "w"), ensure_ascii=False, indent=1)

NOVELTY = f"{ROOT}/novelty_samples.json"
!python {CDC}/scripts/select_novelty_subset.py --samples {TABLE9} --documents {DOCUMENTS} \
    --n_questions {N_NOVELTY} --max_total_pages {MAX_TOTAL_PAGES} --out {NOVELTY}
# clean page screenshots of these documents, then their degraded copies (deterministic, skips existing files)
!python {CDC}/scripts/prepare_subset.py --all --samples {NOVELTY} --documents {DOCUMENTS} --out_root {ROOT} \
    --dataset_name {DATASET} --samples_name _rendered_novelty.json --workers 4
!python {CDC}/scripts/make_variants.py --out_root {ROOT} --dataset_name {DATASET} --variants {DEG_STR} --samples {NOVELTY}

## 4. PREPARE stage: MinerU on the degraded pages, detector calibration, packing
(Skipped automatically in the run stage.)

In [ ]:
if STAGE == "prepare":
    for v in DEGRADED:
        print("=" * 20, v)
        !python {CDC}/scripts/parse_mineru.py --root {ROOT}/{v} --dataset_name {DATASET} --batch_size 100

In [ ]:
if STAGE == "prepare":
    # Calibrate the degradation detector on 10 OTHER documents (none of the 200 Table 9 questions' documents)
    !python {CDC}/scripts/prepare_subset.py --samples {SAMPLES} --documents {DOCUMENTS} --out_root {CALIB_ROOT} \
        --dataset_name {DATASET} --max_docs 10 --max_pages 30 --seed 1 --exclude_samples {TABLE9} --workers 4
    !python {CDC}/scripts/make_variants.py --out_root {CALIB_ROOT} --dataset_name {DATASET} --variants {CALIB_STR}
    !python {CDC}/scripts/calibrate_cdc.py --variants_root {CALIB_ROOT} --dataset_name {DATASET} \
        --variants {CALIB_STR} --out {RESULTS}/calibration.json

In [ ]:
if STAGE == "prepare":
    TAR = f"{WORK}/novelty_variants.tar"
    page_jpeg = re.compile(r"\d+\.jpeg")
    n = 0
    with tarfile.open(TAR, "w") as t:      # MinerU output only; the page screenshots are re-created in the run stage
        for v in DEGRADED:
            for f in sorted(Path(f"{ROOT}/{v}").rglob("*")):
                if not f.is_file() or "_mineru_stage" in f.parts:
                    continue
                if f.parent.parent.name == "documents" and page_jpeg.fullmatch(f.name):
                    continue
                t.add(f, arcname=str(f.relative_to(WORK))); n += 1
    parsed = {v: len(glob.glob(f"{ROOT}/{v}/data/{DATASET}/documents/*/MinerU_Page*/*.md")) for v in DEGRADED}
    print(f"{TAR}: {n} files, {os.path.getsize(TAR) / 1e9:.2f} GB; parsed pages per variant: {parsed}")
    # keep only the tar and the calibration in /kaggle/working so "Save output" stays small
    for d in (ROOT, CALIB_ROOT, f"{DOCLENS}/data"):
        shutil.rmtree(d, ignore_errors=True)
    !du -sh {WORK}
    print("PREPARE DONE -> Save Version -> Quick Save, with 'Save output for this version' ON. Then set STAGE = 'run'.")

## 5. RUN stage: restore, start Qwen3-VL-8B, pilot

In [ ]:
if STAGE == "run":
    tar_var = find_input("novelty_variants.tar")
    assert tar_var, "Add this notebook's saved PREPARE version (novelty_variants.tar) as an input."
    if not glob.glob(f"{ROOT}/{DEGRADED[0]}/data/{DATASET}/documents/*/MinerU_Page*"):
        !tar -xf {tar_var} -C {WORK}
    CALIB = find_input("calibration.json")
    REUSE = find_input("qwen_doclens.jsonl")
    print("calibration:", CALIB)
    print("Table 9 DocLens run to reuse for clean pages:", REUSE or "not attached (clean pages will be recomputed)")
    for v in VARIANTS:
        d = f"{ROOT}/{v}/data/{DATASET}/documents"
        print(v, len(glob.glob(f"{d}/*/*.jpeg")), "pages,", len(glob.glob(f"{d}/*/MinerU_Page*/*.md")), "parsed")

In [ ]:
if STAGE == "run":
    import urllib.request
    log = open(f"{WORK}/vllm.log", "w")
    server = subprocess.Popen([
        "vllm", "serve", "Qwen/Qwen3-VL-8B-Instruct",
        "--tensor-parallel-size", "2", "--dtype", "half",
        "--max-model-len", str(MAX_MODEL_LEN), "--gpu-memory-utilization", "0.92",
        "--limit-mm-per-prompt", json.dumps({"image": MAX_IMAGES}),
        "--mm-processor-kwargs", json.dumps({"max_pixels": 1003520}),
        "--max-num-seqs", "16", "--enable-prefix-caching", "--enforce-eager",
        "--disable-custom-all-reduce", "--port", "8000",
    ], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)  # survives a Stop/Interrupt
    start = time.time()
    while True:
        try:
            urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
            print(f"Server ready after {(time.time() - start) / 60:.1f} min"); break
        except Exception:
            if server.poll() is not None or time.time() - start > 40 * 60:
                print(open(f"{WORK}/vllm.log").read()[-6000:])
                raise RuntimeError("vLLM did not start: send the log above.")
            time.sleep(20)

In [ ]:
if STAGE == "run":
    from reproduction.experiments import EXPERIMENTS, run_args
    args = run_args(dict(EXPERIMENTS["qwen_doclens"], mode="analyze"))   # Te = Ta = 8, same backbone everywhere
    common = (f"python {CDC}/run_cdc.py --doclens_root {DOCLENS} --dataset_name {DATASET} --samples {NOVELTY} "
              f"--max_concurrent {MAX_CONCURRENT} --qwen_base_url http://localhost:8000/v1 "
              f"--qwen_max_images {MAX_IMAGES} --nav_chunk_pages {NAV_CHUNK} "
              f"{'--paper_prompts' if PAPER_PROMPTS else ''} {f'--calibration {CALIB}' if CALIB else ''} {args}")
    v = DEGRADED[0]
    t0 = time.time()
    !{common} --data_root {ROOT}/{v} --variant {v} --limit {PILOT_QUESTIONS} --output {RESULTS}/pilot_{v}.jsonl
    per_q = (time.time() - t0) / PILOT_QUESTIONS
    rows = [json.loads(l) for l in open(f"{RESULTS}/pilot_{v}.jsonl")]
    for r in rows:
        print(r.get("cdc_decision"), f"conf={r.get('cdc_confidence', 0):.2f}", "| gold:", r.get("answer"),
              "| CDC:", r.get("final_prediction"), "| DocLens:", r.get("doclens_prediction"), "| error:", "cdc_error" in r)
    print(f"~{per_q / 60:.1f} min per degraded question -> upper bound {per_q * N_NOVELTY * len(DEGRADED) / 3600:.1f} h "
          f"for {len(DEGRADED)} degraded variants (clean is much cheaper when the Table 9 run is reused)")

## 6. RUN stage: the novelty experiments
Every question runs the Lens Module once, the CDC check, and then the abstain path, the correction path and the original
DocLens path, so the gate can be evaluated at any threshold offline. Clean pages reuse the Table 9 DocLens run.

In [ ]:
if STAGE == "run":
    for v in VARIANTS:
        print("=" * 30, v)
        reuse = f"--reuse_from {REUSE}" if (v == "clean" and REUSE) else ""
        !{common} --data_root {ROOT}/{v} --variant {v} {reuse} --output {RESULTS}/{v}_analyze.jsonl

## 7. Results for the paper

In [ ]:
if STAGE == "run":
    files = " ".join(f"{RESULTS}/{v}_analyze.jsonl" for v in VARIANTS if os.path.exists(f"{RESULTS}/{v}_analyze.jsonl"))
    !python {CDC}/scripts/analyze_results.py {files} --out_dir {RESULTS}/report --dev_frac 0.3
    from IPython.display import Markdown, Image, display
    display(Markdown(open(f"{RESULTS}/report/report.md").read()))
    if os.path.exists(f"{RESULTS}/report/threshold_sweep.png"):
        display(Image(f"{RESULTS}/report/threshold_sweep.png"))

## 8. Save
*Save Version → Quick Save → "Save output for this version"*. Before that, free the space (run the cell below with
`CLEANUP = True`): only `results_novelty/` needs to be kept.

In [ ]:
CLEANUP = False
if CLEANUP:
    for d in (ROOT, CALIB_ROOT, f"{DOCLENS}/data"):
        shutil.rmtree(d, ignore_errors=True)
!du -sh {WORK}